## semantic chunking

In [20]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough, 
 
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Load environment variables
load_dotenv()

True

In [2]:
#first split document into sentences
document = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

In [3]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:01<00:00, 65.04it/s]


In [ ]:
sentences = [s.strip() for s in document.split("\n") if s.strip()]

['LangChain is a framework for building applications with LLMs.',
 'Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.',
 'You can create chains, agents, memory, and retrievers.',
 'The Eiffel Tower is located in Paris.',
 'France is a popular tourist destination.']

In [6]:
# second step is to create embeddings for each sentence
sentence_embeddings = embedding_model.encode(sentences)

In [7]:
thr = 0.7
chunks = []
current_chunks = [sentences[0]]

for i in range(1, len(sentences)):
    sim = cosine_similarity([sentence_embeddings[i]], [sentence_embeddings[i-1]])[0][0]
    if sim > thr:
        current_chunks.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunks))
        current_chunks = [sentences[i]]

In [8]:
chunks

['LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.',
 'You can create chains, agents, memory, and retrievers.',
 'The Eiffel Tower is located in Paris.']

## Create RAG Pipeline

In [11]:
# libraries
from langchain_core.documents import Document

In [16]:
class SemanticChunker:
    def __init__(self, embedding_model= SentenceTransformer('all-MiniLM-L6-v2'), threshold = 0.7):
        self.embedding_model = embedding_model
        self.threshold = threshold
    
    def split(self, document):
        sentences = [s.strip() for s in document.split("\n") if s.strip()]
        sentence_embeddings = self.embedding_model.encode(sentences)

        chunks = []
        current_chunks = [sentences[0]]
        for i in range(1, len(sentences)):
            sim = cosine_similarity([sentence_embeddings[i]], [sentence_embeddings[i-1]])[0][0]
            if sim > self.threshold:
                current_chunks.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunks))
                current_chunks = [sentences[i]]
        chunks.append(". ".join(current_chunks)+".")
        return chunks
    
    def split_documents(self, docs):
        result = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(
                    Document(
                        page_content=chunk,
                        metadata=doc.metadata
                    )
                )
        return result

              

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1061.33it/s]


In [17]:
#first split document into sentences
document = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=document, metadata={"source": "test"})

In [18]:
doc

Document(metadata={'source': 'test'}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [ ]:
# chunk document
chunker = SemanticChunker()
chunks = chunker.split_documents([doc])
chunks

[Document(metadata={'source': 'test'}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={'source': 'test'}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={'source': 'test'}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={'source': 'test'}, page_content='France is a popular tourist destination..')]

In [26]:
# vector database FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embedding)

retriever = vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3216.37it/s]


In [ ]:
#prompt template

template = """
Answer the question based on context of information from a document. If you don't know the answer, say you don't know.
context: {context}
question: {question}
"""

prompt = PromptTemplate.from_template(template)

In [28]:
#llm ollama phi 3
from langchain_community.llms import Ollama
llm = Ollama(model="phi3")

C:\Users\Marawan Ragab\AppData\Local\Temp\ipykernel_31932\3607920452.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="phi3")


In [31]:
# LCEL 
from langchain_core.runnables import RunnableMap
rag_chain = (
    RunnableMap(
        {
            "context": lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"]
        }   
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [32]:
# query
rag_chain.invoke({"question": "What is LangChain used for?"})

'Based on the provided context, LangChain is a framework utilized for developing applications that incorporate Large Language Models (LLMs). It offers modular abstractions allowing integration of LLMs with various tools like OpenAI and Pinecone. The given documents do not offer additional details about its specific uses or features beyond this general description.'

## Semantic chunker with langchain

In [35]:
from langchain_experimental.text_splitter import SemanticChunker


In [36]:
loader = TextLoader("langchain_intro.txt")
docs = loader.load()
docs

[Document(metadata={'source': 'langchain_intro.txt'}, page_content='LangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.')]

In [37]:
chunker = SemanticChunker(embedding)
chunks = chunker.split_documents(docs)
chunks

[Document(metadata={'source': 'langchain_intro.txt'}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.'),
 Document(metadata={'source': 'langchain_intro.txt'}, page_content='The Eiffel Tower is located in Paris. France is a popular tourist destination.')]